In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import json 

In [2]:
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

#Loading the data
df = pd.read_csv('/Users/tahminasahar/earthquake-risk-afghanistan/data/usgs_afghanistan_raw.csv')
df['time'] = pd.to_datetime(df['time'])
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month

print(f" Loaded {len(df)} earthquakes")
print(f"Columns: {df.columns.tolist()}")

 Loaded 8250 earthquakes
Columns: ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource', 'year', 'month']


In [3]:
# Feature 1: Depth Category
# Categorize each earthquake by how deep it was
depth_bins = [0, 70, 300, 800]
depth_labels = ['Shallow', 'Intermediate', 'Deep']
df['depth_category'] = pd.cut(df['depth'], bins=depth_bins, labels=depth_labels, right=False)

# Feature 2: Depth Risk Score
# Convert depth category into a NUMBER for the ML model
# ML models need numbers — they can't understand words like "Shallow"
depth_risk_map = {
    'Shallow': 3,       # Shallow earthquakes are more dangerous
    'Intermediate': 2,  # Intermediate depth is less dangerous
    'Deep': 1          # Deep earthquakes are least dangerous
}

df['depth_risk_score'] = df['depth_category'].map(depth_risk_map)

print("Depth categories created:")
print(df['depth_category'].value_counts())
print(f"\nDepth risk score sample:")
print(df[['depth', 'depth_category', 'depth_risk_score']].head(10))

Depth categories created:
depth_category
Intermediate    4764
Shallow         3479
Deep               7
Name: count, dtype: int64

Depth risk score sample:
     depth depth_category depth_risk_score
0  224.419   Intermediate                2
1  114.000   Intermediate                2
2  223.091   Intermediate                2
3  143.537   Intermediate                2
4   10.000        Shallow                3
5   64.259        Shallow                3
6  102.823   Intermediate                2
7  178.480   Intermediate                2
8  113.421   Intermediate                2
9  209.319   Intermediate                2


In [4]:
#Feature 3
mag_bins = [4.0, 4.9, 5.9, 6.9, 10.0]
mag_labels = ['Light', 'Moderate', 'Strong', 'Major']

df['mag_category'] = pd.cut(
    df['mag'],
    bins=mag_bins,
    labels=mag_labels,
    right=False
)

#  FEATURE 4: Magnitude Risk Score 
mag_risk_map = {
    'Light':    1,
    'Moderate': 2,
    'Strong':   3,
    'Major':    4
}
df['mag_risk_score'] = df['mag_category'].map(mag_risk_map)

#FEATURE 5: Energy Proxy 
df['energy_proxy'] = 10 ** (1.5 * df['mag'])

print("✅ Magnitude features created!")
print("\nEarthquakes per magnitude category:")
print(df['mag_category'].value_counts())
print("\nSee the energy difference — notice how huge it gets:")
print(df[['mag', 'mag_category', 'energy_proxy']].head(8))


✅ Magnitude features created!

Earthquakes per magnitude category:
mag_category
Light       7404
Moderate     763
Strong        75
Major          8
Name: count, dtype: int64

See the energy difference — notice how huge it gets:
   mag mag_category  energy_proxy
0  4.0        Light  1.000000e+06
1  4.2        Light  1.995262e+06
2  4.0        Light  1.000000e+06
3  4.1        Light  1.412538e+06
4  4.7        Light  1.122018e+07
5  4.3        Light  2.818383e+06
6  4.3        Light  2.818383e+06
7  4.1        Light  1.412538e+06


In [6]:
#Distance From Kabul 
KABUL_LAT = 34.5553
KABUL_LON = 69.2075

# Haversine formula — real distance on a curved Earth
def haversine_distance(lat1, lon1, lat2, lon2):
    
    # Step 1: Convert degrees to radians
    # Why? Math functions (sin, cos) need radians not degrees
    # Degrees are for humans, radians are for math
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    # Step 2: Calculate the differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Step 3: Apply the Haversine formula
    # This accounts for Earth's spherical shape
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Step 4: Multiply by Earth's radius to get km
    earth_radius_km = 6371
    return c * earth_radius_km

# Apply to every earthquake row
df['dist_to_kabul'] = haversine_distance(
    df['latitude'],   # each earthquake's latitude
    df['longitude'],  # each earthquake's longitude
    KABUL_LAT,        # Kabul's latitude
    KABUL_LON         # Kabul's longitude
)
print(df[['latitude', 'longitude', 'place', 'dist_to_kabul']].head(5))

   latitude  longitude                                place  dist_to_kabul
0   36.7412    70.9153       15 km SSE of Jurm, Afghanistan     287.894765
1   36.1980    70.6585       75 km SSW of Jurm, Afghanistan     225.095064
2   36.5397    71.2467  30 km WSW of Ashkāsham, Afghanistan     287.603205
3   36.7477    71.3742  15 km WNW of Ashkāsham, Afghanistan     312.639348
4   31.1844    66.6548        35 km NNE of Chaman, Pakistan     444.176160


In [ ]:
# FEATURE 7: Grid Zone (being in the same zone)
df['lat_grid'] = df['latitude'].round(1)
df['lon_grid'] = df['longitude'].round(1)

print("✅ Geographic features created!")
print(f"\nAverage earthquake distance from Kabul: {df['dist_to_kabul'].mean():.0f} km")
print(f"Closest earthquake to Kabul: {df['dist_to_kabul'].min():.0f} km")
print(f"Farthest earthquake: {df['dist_to_kabul'].max():.0f} km")

✅ Geographic features created!

Average earthquake distance from Kabul: 351 km
Closest earthquake to Kabul: 10 km
Farthest earthquake: 1051 km


In [9]:
# FEATURE 8: Season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'
df['season'] = df['month'].apply(get_season)
print("✅ Seasonal features created!")
print("\nEarthquakes per season:")
print(df['season'].value_counts())

✅ Seasonal features created!

Earthquakes per season:
season
Fall      2271
Spring    2051
Winter    2000
Summer    1928
Name: count, dtype: int64


In [10]:
#FEATURE 9: Decade
df['decade'] = (df['year'] // 10) * 10
print("\nEarthquakes by decade:")
print(df['decade'].value_counts().sort_index())


Earthquakes by decade:
decade
1990    1755
2000    2648
2010    2378
2020    1469
Name: count, dtype: int64


In [ ]:
#FEATURE 10: Days Since Start
earliest = df['time'].min()
df['days_since_start'] = (df['time'] - earliest).dt.days


## Normalization 


In [14]:
from sklearn.preprocessing import MinMaxScaler
# This is the most important cell of the entire project
# We combine everything into ONE master risk number

scaler = MinMaxScaler()

# Make sure these column names exactly match your DataFrame
risk_columns = ['mag', 'depth_risk_score', 'energy_proxy']

# Normalize all three columns to 0-1 scale
normalized_values = scaler.fit_transform(df[risk_columns])

# Store each normalized column separately
df['mag_normalized']          = normalized_values[:, 0]
df['depth_risk_normalized']   = normalized_values[:, 1]
df['energy_normalized']       = normalized_values[:, 2]

# Combine into one master risk score
df['combined_risk_score'] = (
    df['mag_normalized']        * 0.4  +
    df['energy_normalized']     * 0.35 +
    df['depth_risk_normalized'] * 0.25
)

print("✅ Combined risk score created!")
print(f"\nRisk score statistics:")
print(df['combined_risk_score'].describe())
print(f"\nTop 10 most dangerous earthquakes:")
print(df[['time', 'place', 'mag', 'depth', 'combined_risk_score']]
      .sort_values('combined_risk_score', ascending=False)
      .head(10)
      .to_string())

✅ Combined risk score created!

Risk score statistics:
count    8250.000000
mean        0.223997
std         0.080528
min         0.022224
25%         0.147224
50%         0.202792
75%         0.294449
max         1.000000
Name: combined_risk_score, dtype: float64

Top 10 most dangerous earthquakes:
                                 time                                place  mag  depth  combined_risk_score
5342 2005-10-08 03:50:40.800000+00:00  21 km NNE of Muzaffar?b?d, Pakistan  7.6   26.0             1.000000
2655 2015-10-26 09:09:42.560000+00:00       Hindu Kush region, Afghanistan  7.5  231.0             0.761670
2589 2015-12-07 07:50:05.950000+00:00      104 km W of Murghob, Tajikistan  7.2   22.0             0.693471
6090 2002-03-03 12:08:19.740000+00:00        51 km SW of Jurm, Afghanistan  7.4  225.6             0.678193
7095 1997-02-27 21:08:02.360000+00:00        29 km ESE of Harnai, Pakistan  7.1   33.0             0.656683
620  2023-02-23 00:37:38.163000+00:00     65 km WSW